<a target="_parent" href="https://colab.research.google.com/github/mark-baumann/ART/blob/main/examples/tic_tac_toe/tic_tac_toe_wandb.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Tic-Tac-Toe mit ART & Weights & Biases (Google Colab)

Dieses Notebook ist die **Google-Colab-Variante von
`examples/tic_tac_toe/tic-tac-toe.py`** aus dem
[ART-Repository](https://github.com/mark-baumann/ART). Der Code folgt der
`.py`-Datei so eng wie möglich – nur in einzelne Zellen aufgeteilt und an
Colab angepasst (kleineres Basismodell + weniger Steps, damit es auf einer
kostenlosen T4-GPU durchläuft).

Dabei lernst du **Weights & Biases** kennen: ART loggt den Trainingsfortschritt
automatisch nach W&B – `model.log(...)` schreibt die Metriken in einen
W&B-`Run`, und die `@weave.op`-Rollouts erscheinen als **Weave-Traces** im
selben Projekt.

## Aufbau

1. Laufzeit prüfen (GPU)
2. ART + Backend installieren
3. W&B einrichten (Login, Entity, Projekt)
4. Spiel-Logik (`game_utils.py`) und Rollout (`rollout.py`) bereitstellen
5. Modell registrieren (LocalBackend) – wie in `tic-tac-toe.py`
6. Trainings-Loop – wie in `tic-tac-toe.py`, aber mit W&B-Tracking
7. Ergebnisse in W&B lesen

> Zum Vergleich: Öffne im Repo die Datei `examples/tic_tac_toe/tic-tac-toe.py`
> neben diesem Notebook – die Abschnitte 4–6 entsprechen ihr fast 1:1.


## 1. Laufzeit prüfen (GPU empfohlen)

In Colab: **Laufzeit → Laufzeittyp ändern → T4 GPU**.


In [ ]:
import os, sys, subprocess

def is_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

IN_COLAB = is_colab()
print("Google Colab:", IN_COLAB)

try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print("GPU:", out.stdout.strip() or "(keine GPU)")
    GPU_OK = out.returncode == 0
except Exception:
    GPU_OK = False
    print("GPU: nicht verfügbar")

print("Python:", sys.version.split()[0])
if IN_COLAB and not GPU_OK:
    print("Tipp: Laufzeit -> Laufzeittyp ändern -> T4 GPU aktivieren.")


## 2. ART + Backend installieren

Installiert `openpipe-art[backend]` (unsloth/vLLM/torch) in der im Repo
deklarierten Version `0.5.18` – wie sie `tic-tac-toe.py` erwartet.
Falls die gepinnte Version nicht installierbar ist, wird automatisch die
neueste probiert. **Dauert einige Minuten und braucht eine GPU-Laufzeit.**


In [ ]:
import subprocess, sys

if GPU_OK:
    rc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
         "openpipe-art[backend]==0.5.18", "--prerelease", "allow"],
    ).returncode
    if rc != 0:
        print("Pinned-Version fehlgeschlagen -> neueste Version probieren ...")
        rc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
             "openpipe-art[backend]", "--prerelease", "allow"],
        ).returncode
    print("Fertig." if rc == 0 else f"Install fehlgeschlagen (Code {rc}).")
else:
    print("Keine GPU -> Installation übersprungen (bitte T4-GPU aktivieren).")


## 3. Weights & Biases einrichten

ART schreibt Metriken automatisch in einen W&B-Run, sobald
`WANDB_API_KEY` gesetzt ist (`model.log(...)`). Zusätzlich nutzen wir Weave
für die Rollout-Traces.

1. API-Key unter **wandb.ai/settings → API keys** erzeugen.
2. Unten einfügen. `ENTITY`/`PROJECT` bitte auf deine Werte stellen
   (z. B. `ENTITY=augustinum`, `PROJECT=init` aus deinem Link
   https://wandb.ai/augustinum/init).


In [ ]:
import getpass

API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass.getpass("Bitte W&B-API-Key eingeben (oder leer lassen): ").strip()
    os.environ["WANDB_API_KEY"] = API_KEY

ENTITY  = os.environ.get("WANDB_ENTITY", "augustinum")
PROJECT = os.environ.get("WANDB_PROJECT", "tic-tac-toe")
os.environ["WANDB_ENTITY"] = ENTITY
os.environ["WANDB_PROJECT"] = PROJECT
print("Entity :", ENTITY)
print("Project:", PROJECT)


In [ ]:
import wandb

if os.environ.get("WANDB_API_KEY"):
    wandb.login()
    print("W&B Login OK")
else:
    print("Kein API-Key -> Metriken werden lokal (offline) geschrieben.")


## 4. Spiel-Logik & Rollout bereitstellen

Die nächsten beiden Zellen schreiben die **Original-Module** aus
`examples/tic_tac_toe/` (unverändert) ins Arbeitsverzeichnis, damit das
Notebook in Colab ohne das Repo auskommt:

- `game_utils.py` – Brett, Regeln, Siegerprüfung
- `rollout.py` – der eigentliche ART-Rollout (`TicTacToeScenario`, `rollout`)

> `tic-tac-toe.py` importiert diese Module genauso (`from rollout import ...`).


In [ ]:
%%writefile game_utils.py
import random
from typing import Literal, TypedDict
import xml.etree.ElementTree as ET


class TicTacToeGame(TypedDict):
    board: list[list[str]]
    agent_symbol: Literal["x", "o"]
    opponent_symbol: Literal["x", "o"]


def generate_game(board_length: int = 3) -> TicTacToeGame:
    board = [["_" for _ in range(board_length)] for _ in range(board_length)]
    agent_symbol = random.choice(["x", "o"])
    opponent_symbol = "x" if agent_symbol == "o" else "o"
    return {
        "board": board,
        "agent_symbol": agent_symbol,
        "opponent_symbol": opponent_symbol,
    }


def render_board(game: TicTacToeGame) -> str:
    board = game["board"]
    board_length = len(board)
    # print something like this:
    #    1   2   3
    # A  _ | x | x
    # B  o | _ | _
    # C  _ | o | _
    # where _ is an empty cell

    board_str = "   " + "   ".join([str(i + 1) for i in range(board_length)]) + "\n"
    for i in range(board_length):
        board_str += f"{chr(65 + i)}  {board[i][0]} | {board[i][1]} | {board[i][2]}\n"
    return board_str


def get_opponent_move(game: TicTacToeGame) -> tuple[int, int]:
    # get a random empty cell
    empty_cells = [
        (i, j) for i in range(3) for j in range(3) if game["board"][i][j] == "_"
    ]
    return random.choice(empty_cells)


def apply_agent_move(game: TicTacToeGame, move: str) -> None:
    board_length = len(game["board"])

    try:
        root = ET.fromstring(move)
        square = root.text
    except Exception:
        raise ValueError("Invalid xml")

    try:
        row_index = ord(square[0]) - 65
        col_index = int(square[1]) - 1
    except Exception as e:
        print(e)
        raise ValueError("Unable to parse square")

    if (
        row_index < 0
        or row_index >= board_length
        or col_index < 0
        or col_index >= board_length
    ):
        raise ValueError(
            f"Invalid move, row or column out of bounds: {row_index}, {col_index}"
        )

    # check if the move is valid
    if game["board"][row_index][col_index] != "_":
        raise ValueError("Square already occupied")

    game["board"][row_index][col_index] = game["agent_symbol"]


def check_winner(board: list[list[str]]) -> Literal["x", "o", "draw", None]:
    board_length = len(board)
    # check rows
    for row in board:
        if row.count(row[0]) == board_length and row[0] != "_":
            return row[0]
    # check columns
    for col in range(board_length):
        if [board[row][col] for row in range(board_length)].count(
            board[0][col]
        ) == board_length and board[0][col] != "_":
            return board[0][col]

    # top right to bottom left
    upward_diagonal = [board[i][board_length - i - 1] for i in range(board_length)]
    if (
        upward_diagonal.count(upward_diagonal[0]) == board_length
        and upward_diagonal[0] != "_"
    ):
        return upward_diagonal[0]

    # top left to bottom right
    downward_diagonal = [board[i][i] for i in range(board_length)]
    if (
        downward_diagonal.count(downward_diagonal[0]) == board_length
        and downward_diagonal[0] != "_"
    ):
        return downward_diagonal[0]

    # check for draw
    if all(cell != "_" for row in board for cell in row):
        return "draw"
    return None


In [ ]:
%%writefile rollout.py
import math
import os
import time

from dotenv import load_dotenv
from game_utils import (
    apply_agent_move,
    check_winner,
    generate_game,
    get_opponent_move,
    render_board,
)
import openai
from pydantic import BaseModel
import weave

import art

load_dotenv()


class TicTacToeScenario(BaseModel):
    step: int


@weave.op
@art.retry(exceptions=(openai.LengthFinishReasonError,))
async def rollout(model: art.Model, scenario: TicTacToeScenario) -> art.Trajectory:
    game = generate_game()

    trajectory = art.Trajectory(
        messages_and_choices=[
            {
                "role": "system",
                "content": f"You are a tic-tac-toe player. You are playing against an opponent. Always choose the move most likely to lead to an eventual win. Return your move as an XML object with a single property 'move', like so: <move>A1</move>. Optional moves are 'A1', 'B3', 'C2', etc. You are the {game['agent_symbol']} symbol.",
            }
        ],
        reward=0,
    )

    move_number = 0
    invalid_move = False

    if game["agent_symbol"] == "o":
        starting_opponent_move = get_opponent_move(game)
        game["board"][starting_opponent_move[0]][starting_opponent_move[1]] = game[
            "opponent_symbol"
        ]

    while check_winner(game["board"]) is None:
        trajectory.messages_and_choices.append(
            {"role": "user", "content": render_board(game)}
        )

        messages = trajectory.messages()

        try:
            client = model.openai_client()
            chat_completion = await client.chat.completions.create(
                model=model.get_inference_name(),
                messages=messages,
                max_completion_tokens=128,
            )
        except openai.LengthFinishReasonError as e:
            raise e
        except Exception as e:
            print("caught exception generating chat completion")
            print(e)
            global failing_trajectory
            failing_trajectory = trajectory
            raise e

        choice = chat_completion.choices[0]
        content = choice.message.content
        assert isinstance(content, str)
        trajectory.messages_and_choices.append(choice)

        try:
            apply_agent_move(game, content)
        except ValueError:
            invalid_move = True
            trajectory.reward = -100 + (math.log(move_number + 1) / math.log(100))
            break

        move_number += 1
        if check_winner(game["board"]) is not None:
            break

        opponent_move = get_opponent_move(game)
        game["board"][opponent_move[0]][opponent_move[1]] = game["opponent_symbol"]

    winner = check_winner(game["board"])

    if winner == game["agent_symbol"]:
        trajectory.reward = 1
        trajectory.metrics["win"] = 1
    elif winner == game["opponent_symbol"]:
        trajectory.reward = 0
        trajectory.metrics["win"] = 0
    elif winner == "draw":
        trajectory.reward = 0.5
        trajectory.metrics["win"] = 0.5

    trajectory.metrics["num_moves"] = move_number
    trajectory.metrics["invalid_move"] = 1 if invalid_move else 0

    return trajectory


## 5. Modell registrieren (LocalBackend)

Entspricht dem Anfang von `main()` in `tic-tac-toe.py`:

```python
backend = LocalBackend()
model = art.TrainableModel(name=..., project=..., base_model=...)
await model.register(backend)
```

Abweichung für Colab: Als Basismodell nehmen wir `Qwen/Qwen2.5-3B-Instruct`
(statt des 8B-Llama aus der `.py`, das auf einer T4 nicht läuft). Zum
Vergleichen einfach `BASE_MODEL` ändern (braucht dann mehr VRAM / ggf.
`HF_TOKEN`).


In [ ]:
if GPU_OK:
    import nest_asyncio
    nest_asyncio.apply()

    import random
    from rollout import TicTacToeScenario, rollout
    import art
    from art.local.backend import LocalBackend
    from art.utils.strip_logprobs import strip_logprobs

    random.seed(42)

    # Weave = W&B-Tracing (identisch zur weave.init-Zeile in tic-tac-toe.py)
    if os.environ.get("WANDB_API_KEY"):
        import weave
        try:
            weave.init(f"{ENTITY}/{PROJECT}" if ENTITY else PROJECT,
                       global_postprocess_output=strip_logprobs)
            print("Weave initialisiert.")
        except Exception as e:
            print("weave.init fehlgeschlagen (optional):", e)

    backend = LocalBackend(path="./.art")

    BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # tic-tac-toe.py: meta-llama/Meta-Llama-3.1-8B-Instruct
    MODEL_NAME = "001-wandb-colab"

    model = art.TrainableModel(
        name=MODEL_NAME,
        project=PROJECT,
        base_model=BASE_MODEL,
    )

    await model.register(backend)
    print("Modell registriert. Inference:", model.inference_base_url)
else:
    print("GPU fehlt -> übersprungen.")


## 6. Training-Loop

Der Loop entspricht `main()` aus `tic-tac-toe.py`:

```python
for i in range(await model.get_step(), STEP):
    train_groups = await art.gather_trajectory_groups(... rollout(...) x ROLLOUTS ...)
    await model.delete_checkpoints()
    result = await backend.train(model, train_groups, learning_rate=LEARNING_RATE)
    await model.log(train_groups, metrics=result.metrics, step=result.step, split="train")
```

**Colab-Anpassung:** `STEP` und `ROLLOUTS_PER_STEP` sind reduziert, damit es in
einer kostenlosen Session durchläuft. Zum echten Training (wie in der `.py`)
beides erhöhen.


In [ ]:
if GPU_OK:
    STEP = 2                 # tic-tac-toe.py: 50
    ROLLOUTS_PER_STEP = 32   # tic-tac-toe.py: 96
    LEARNING_RATE = 5e-5

    print("W&B-Run:",
          f"https://wandb.ai/{ENTITY or '<entity>'}/{PROJECT}/runs/{model.name}")

    for i in range(await model.get_step(), STEP):
        print(f"--- Step {i} ---")
        train_groups = await art.gather_trajectory_groups(
            (
                art.TrajectoryGroup(
                    rollout(model, TicTacToeScenario(step=i))
                    for _ in range(ROLLOUTS_PER_STEP)
                )
                for _ in range(1)
            ),
            pbar_desc="gather",
            max_exceptions=1,
        )
        await model.delete_checkpoints()
        result = await backend.train(model, train_groups, learning_rate=LEARNING_RATE)
        await model.log(
            train_groups, metrics=result.metrics, step=result.step, split="train"
        )

    print("Training abgeschlossen.")
else:
    print("GPU fehlt -> übersprungen.")


## 7. Ergebnisse in W&B lesen

Während des Trainings findest du in deinem Projekt
(`https://wandb.ai/<entity>/<project>`):

| Bereich | Was du siehst |
| --- | --- |
| **Runs** | der ART-Training-Run (`001-wandb-colab`) |
| **Charts** | `reward`, `train/*`, `data/*`, Loss-Kurven (x-Achse = `training_step`) |
| **Weave** | die Rollout-Traces der `@weave.op`-Calls |
| **System** | GPU-Auslastung, Speicher |
| **Files** | `history.jsonl`, Parquet-Trajectories, Checkpoints |


In [ ]:
if GPU_OK:
    try:
        await backend.close()
        print("Backend geschlossen.")
    except Exception as e:
        print("Backend-Close übersprungen:", e)

print("Projekt öffnen:",
      f"https://wandb.ai/{ENTITY or '<entity>'}/{PROJECT}/")


## Zusammenfassung

- Das Notebook ist eine enge 1:1-Übertragung von `examples/tic_tac_toe/tic-tac-toe.py`.
- Statt der `.py`-Defaults (`meta-llama/Meta-Llama-3.1-8B-Instruct`, `STEP=50`,
  96 Rollouts) nutzt es für Colab `Qwen/Qwen2.5-3B-Instruct`, `STEP=2` und 32
  Rollouts – sonst ist der Ablauf identisch.
- ART loggt automatisch nach W&B: `model.log(...)` → Run-Metriken,
  `@weave.op`-Rollouts → Weave-Traces.

Viel Spaß beim Experimentieren! 🚀
